# TabAnywhere Gemma 4 E4B Unsloth Fine-Tuning

This notebook trains the unified rewrite-based edit prediction model with Unsloth QLoRA, then exports GGUF variants for the macOS app.

Recommended runtime: **Colab Pro L4 or A100**. A free T4 can be used for a smoke test with small data, but it may be slow or memory-constrained.

Before running, set `REPO_URL` below to a GitHub URL containing this repo. If the repo is private, create a Colab secret named `GITHUB_TOKEN` with read access.

In [ ]:
#@title 1. Check GPU
!nvidia-smi

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
#@title 2. Clone repo
REPO_URL = "https://github.com/Subjective/autocomplete"  # e.g. "https://github.com/YOUR_ORG/autocomplete.git"
BRANCH = ""    # optional; leave empty for default branch

assert REPO_URL, 'Set REPO_URL before running this cell.'

import os
from pathlib import Path

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None

clone_url = REPO_URL
if token and REPO_URL.startswith('https://github.com/'):
    clone_url = REPO_URL.replace('https://', f'https://{token}@', 1)

%cd /content
!rm -rf autocomplete
branch_arg = f"--branch {BRANCH}" if BRANCH else ""
!git clone {branch_arg} {clone_url} autocomplete
%cd /content/autocomplete

In [ ]:
#@title 3. Install training dependencies
!pip install -U pip
!pip install -r training/requirements-training.txt

# Runtime sanity check
import unsloth, torch
print('Unsloth imported')
print('Torch:', torch.__version__)

In [ ]:
#@title 4. Optional Hugging Face login
# If google/gemma-4-E4B-it is gated, create a Colab secret named HF_TOKEN.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print('Logged in to Hugging Face')
else:
    print('No HF_TOKEN secret found; continuing unauthenticated.')

In [ ]:
#@title 5. Validate and prepare data
!python3 training/scripts/validate_dataset.py training/data/seed/tabanywhere_seed_v001.jsonl --kind sft
!python3 training/scripts/validate_dataset.py training/data/eval/tabanywhere_eval_v001.jsonl --kind eval
!python3 training/scripts/validate_dataset.py training/data/preference/tabanywhere_dpo_v001.jsonl --kind preference
!python3 training/scripts/split_dataset.py training/data/seed/tabanywhere_seed_v001.jsonl --out-dir training/runs/splits
!python3 training/scripts/prepare_sft_dataset.py training/data/seed/tabanywhere_seed_v001.jsonl training/runs/seed_messages.jsonl

In [ ]:
#@title 6. Train LoRA and merge
!python3 training/scripts/train_unsloth_qlora.py --config training/configs/sft_gemma_lora.yaml --merge

In [ ]:
#@title 7. Export GGUF variants
!python3 training/scripts/export_gguf_unsloth.py --config training/configs/export_gguf.yaml
!find training/models -maxdepth 3 -type f | sort

In [ ]:
#@title 8. Save artifacts to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/tabanywhere-models
!cp -R training/models/* /content/drive/MyDrive/tabanywhere-models/
!find /content/drive/MyDrive/tabanywhere-models -maxdepth 3 -type f | sort